# 03 · LangGraph in production: checkpoints, threads, interrupts and resuming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/agent-frameworks/03_langgraph_production.ipynb) &nbsp; 📖 Concept deck: [LangGraph in production](https://nilesh-salpe.github.io/visual-ai/agent-frameworks/03-langgraph-production/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** per-thread state with a checkpointer, a human approval step with `interrupt()` and `Command(resume=...)`, and resuming a failed run from its last checkpoint without redoing finished steps.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with a scripted fake chat model, so no API key or GPU is needed. Run the **Install** cell first (pinned to LangChain 1.4.6 and LangGraph 1.2.15).

In [ ]:
# @title Install pinned packages
%pip install -q "langchain==1.4.6" "langchain-core==1.6.10" "langgraph==1.2.15"

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

def thread(tid):
    """The config that selects a conversation thread in the checkpointer."""
    return {"configurable": {"thread_id": tid}}

## Task 1: Threads and checkpoints

Build a one-node graph over `CounterState` whose node `bump` adds 1 to `count` (the key's reducer is `operator.add`, so return `{"count": 1}`). Compile it **with an `InMemorySaver` checkpointer**. Each `thread_id` then keeps its own state across `invoke` calls.

In [ ]:
class CounterState(TypedDict):
    count: Annotated[int, operator.add]

def build_counter_app():
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_build_counter_app(f):
    app = f()
    app.invoke({"count": 0}, thread("a")); r = app.invoke({"count": 0}, thread("a"))
    assert r["count"] == 2, f"thread a should remember its count across calls, got {r['count']}"
    assert app.invoke({"count": 0}, thread("b"))["count"] == 1, "thread b starts fresh"
    assert app.get_state(thread("a")).values["count"] == 2
    assert len(list(app.get_state_history(thread("a")))) >= 4, "every step leaves a checkpoint"

check("build_counter_app", _test_build_counter_app, build_counter_app)

## Task 2: Human approval with interrupt()

Build `plan → approve → (execute | cancel) → END` over `ApprovalState`, compiled with a checkpointer:
* `plan` sets `action` to `"delete_branch"`;
* `approve` calls `interrupt({"action": state["action"]})` and stores the returned value in `approved`;
* a conditional edge goes to `execute` (sets `status="executed"` and appends the action to the global `EXECUTED` list) or `cancel` (sets `status="cancelled"`).

⚠️ When resumed, the **whole `approve` node runs again from the top**, so never put the side effect before the `interrupt()` call.

In [ ]:
EXECUTED = []

class ApprovalState(TypedDict):
    action: str
    approved: bool
    status: str

def build_approval_app():
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_build_approval_app(f):
    EXECUTED.clear()
    app = f()
    r = app.invoke({"action": "", "approved": False, "status": ""}, thread("ok"))
    assert "__interrupt__" in r, "the run should pause at the approval step"
    assert r["__interrupt__"][0].value == {"action": "delete_branch"}
    assert app.get_state(thread("ok")).next == ("approve",)
    r = app.invoke(Command(resume=True), thread("ok"))
    assert r["status"] == "executed" and EXECUTED == ["delete_branch"], (r, EXECUTED)
    app.invoke({"action": "", "approved": False, "status": ""}, thread("no"))
    r = app.invoke(Command(resume=False), thread("no"))
    assert r["status"] == "cancelled" and EXECUTED == ["delete_branch"], "a rejected action must not run"

check("build_approval_app", _test_build_approval_app, build_approval_app)

## Task 3: Resume after a crash

With a checkpointer, a failed run keeps the checkpoints of the steps that finished. Invoking the same thread with input `None` **continues from the last checkpoint** instead of starting over. Write `run_with_resume(app, inputs, config, max_restarts=3)`: invoke; on an exception, re-invoke with `None` (up to `max_restarts` times); return `(result, restarts)`. Re-raise if it still fails.

In [ ]:
def run_with_resume(app, inputs, config, max_restarts=3):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_run_with_resume(f):
    calls = {"fetch": 0, "flaky": 0}
    class S(TypedDict):
        log: Annotated[list, operator.add]
    def fetch(s):
        calls["fetch"] += 1
        return {"log": ["fetched"]}
    def flaky(s):
        calls["flaky"] += 1
        if calls["flaky"] < 3:
            raise ConnectionError("upstream timed out")
        return {"log": ["processed"]}
    g = StateGraph(S); g.add_node("fetch", fetch); g.add_node("flaky", flaky)
    g.add_edge(START, "fetch"); g.add_edge("fetch", "flaky"); g.add_edge("flaky", END)
    app = g.compile(checkpointer=InMemorySaver())
    result, restarts = f(app, {"log": []}, thread("job"))
    assert result["log"] == ["fetched", "processed"] and restarts == 2, (result, restarts)
    assert calls["fetch"] == 1, "a finished step must not run again after a restart"
    calls.update(fetch=0, flaky=-10)
    try:
        f(app, {"log": []}, thread("job2"), max_restarts=1)
        assert False, "should re-raise after max_restarts"
    except ConnectionError:
        pass

check("run_with_resume", _test_run_with_resume, run_with_resume)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Threads and checkpoints
def build_counter_app_solution():
    g = StateGraph(CounterState)
    g.add_node("bump", lambda s: {"count": 1})
    g.add_edge(START, "bump"); g.add_edge("bump", END)
    return g.compile(checkpointer=InMemorySaver())

check("build_counter_app (reference)", _test_build_counter_app, build_counter_app_solution)

In [ ]:
# @title Solution for Task 2: Human approval with interrupt()
def build_approval_app_solution():
    g = StateGraph(ApprovalState)
    g.add_node("plan", lambda s: {"action": "delete_branch"})
    g.add_node("approve", lambda s: {"approved": bool(interrupt({"action": s["action"]}))})
    def execute(s):
        EXECUTED.append(s["action"])
        return {"status": "executed"}
    g.add_node("execute", execute)
    g.add_node("cancel", lambda s: {"status": "cancelled"})
    g.add_edge(START, "plan"); g.add_edge("plan", "approve")
    g.add_conditional_edges("approve", lambda s: "execute" if s["approved"] else "cancel")
    g.add_edge("execute", END); g.add_edge("cancel", END)
    return g.compile(checkpointer=InMemorySaver())

check("build_approval_app (reference)", _test_build_approval_app, build_approval_app_solution)

In [ ]:
# @title Solution for Task 3: Resume after a crash
def run_with_resume_solution(app, inputs, config, max_restarts=3):
    restarts = 0
    while True:
        try:
            return app.invoke(inputs if restarts == 0 else None, config), restarts
        except Exception:
            if restarts >= max_restarts:
                raise
            restarts += 1

check("run_with_resume (reference)", _test_run_with_resume, run_with_resume_solution)

---
## Experiments
Every step leaves a checkpoint. Walk a thread's history (newest first): this is what "time travel" and debugging replay are built on.

In [ ]:
app = build_approval_app_solution()
app.invoke({"action": "", "approved": False, "status": ""}, thread("demo"))
app.invoke(Command(resume=True), thread("demo"))
for snap in app.get_state_history(thread("demo")):
    print(f"step {snap.metadata.get('step'):>2}  next={snap.next!s:14} status={snap.values.get('status')!r}")

---
## Takeaways
* A **checkpointer** saves state after every step, per **thread**; this gives memory, resumption after failure, and history.
* `interrupt()` pauses a run until `Command(resume=value)`; the node **re-runs from its start**, so side effects go after the interrupt or in a later node.
* In production use a durable checkpointer (e.g. Postgres) instead of `InMemorySaver`, and make nodes idempotent.